# LTS Rutherford Workflow Explorer

Interactive companion to `scripts/main/main.py`. Lets you:

1. Verify Python deps for this kernel.
2. Pick an existing run folder OR launch a new run with the same flags as the CLI.
3. Run pipeline stages one at a time (each cell has a button + live log tail).
4. Visualise the artefacts each stage produced.

**Stays in sync with main.py.** Every stage button calls a `WorkflowRunner`
method directly â€” no logic is duplicated here. When `main.py` changes, this
notebook picks up the new behaviour automatically.

**First-time setup:** run `setup.ipynb` once to pip-install deps and pull
the two Docker images.

## 0. Dependency check

Verifies the current kernel has every Python package the pipeline needs,
plus the Docker daemon and bundled FreeCAD/ParaView under `tools/`. Doesn't
try to install anything â€” see `setup.ipynb` for that.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

_HERE = Path.cwd() if Path.cwd().name == 'notebooks' else (Path.cwd() / 'notebooks')
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import workflow_viz as v
_summary = v.display_environment_check(include_docker_images=False)
if not _summary['pip']:
    print('!! Some Python packages are missing. Run setup.ipynb (or `pip install -e .[notebook]`) before continuing.')
if not _summary['tools']:
    print('!! Docker or bundled tools are missing. Run setup.ipynb to bootstrap.')

## 0b. Auto-install missing pip dependencies

If the table above flagged any Python packages, run this cell to install
everything `pyproject.toml` lists (including the `notebook` extra). No-op
when nothing is missing, so safe to re-run on every kernel start.

Docker daemon / FreeCAD / ParaView cannot be auto-installed here - run
`setup.ipynb` (or install Docker Desktop manually) if those are missing.

In [ ]:
ok = v.install_missing_pip(extras='notebook')
if ok:
    # Re-display the table so you see the post-install state.
    v.display_environment_check(include_docker_images=False);

## 1. Pick an existing run folder (for viewing or rerun)

Each cable's run folders are listed newest-first. Re-execute downstream cells
after changing the selection.

In [ ]:
rp = v.RunPicker()
rp.display()

## 1b. Design a new cable preset

Interactive panel for laying out a new Rutherford cable: number of strands,
strand diameter, squashed (post-compaction) cable dimensions, transposition
pitch, stack count, and the wire sub-block (Nb3Sn subelement count,
Cu/non-Cu ratio, hex diameter, Nb barrier thickness, bronze fraction).

**Live plot.** Blue circles + dotted outline show the *natural* (uncompressed)
Rutherford arrangement that LS-DYNA sees at t=0. Red dashed rectangle shows
the *squashed* boundary the compaction plates push toward. Sliders update
the plot, JSON preview, and warnings on every change.

**Constraints.** Soft-clipped sliders + warning banner:
* `cable_height` must lie strictly between `D_strand` and `2*D_strand`
  (Rutherford keystone range; outside this means physically impossible or
  not actually compressed).
* Packing factor `N*π*D²/4 / (width*height)` must be ≤ 1.0 (the strand
  area must fit in the cable rectangle); production values typically sit
  at 0.85–0.92.

**Save.** Backs up `cable_parameters_user.json` to `*.bak`, then writes
the new preset under `cables.<NAME>` and updates `active_cable`. After
saving, click the **Refresh** button in the run picker (Section 1) to see
the new cable in the dropdown.

To start from an existing preset instead of the defaults, pass
`initial_from='R2D2_LF'` (or any other cable name) when instantiating
`CableDesigner`.

In [ ]:
designer = v.CableDesigner(name='NEW_CABLE')
# Or start from an existing preset:
# designer = v.CableDesigner(name='R2D2_LF_TWEAK', initial_from='R2D2_LF')
designer.display()

## 2. Configure a new run (mirrors `main.py` CLI flags)

These settings drive every per-stage button below. Read at click-time, so you
can change flags between stages without rebuilding the panel.

The HPC backend works with **any** SSH-reachable SLURM cluster — set the host,
user, and remote base path. ETH Euler defaults are pre-filled; CERN/PSI users
edit in place. Under the hood the widget sets the `HPC_HOST` / `HPC_USER` /
`HPC_REMOTE_BASE` env vars before launching.

In [ ]:
rc = v.RunConfig()
rc.display()

## 3. Stage tracker (current run folder)

In [ ]:
if rp.run_folder is None:
    print('No run folder selected. Use "Create new run folder" below.')
else:
    v.display_stage_tracker(rp.run_folder)

## 4. Pipeline stages

Each stage has a **Run** button + status + an Output panel that captures the
stage's stdout/stderr (logger output included) in real time. Heavy stages
(LS-DYNA, cablestack, compbox) additionally tail the stage's container log
file so you see progress live, AND get a red **Stop containers** button
that calls `docker compose down` on every container project for the current
run folder.

Stages run in a background thread; the notebook UI stays responsive. To
abort a running stage:

* **Stop containers** (red button next to the heavy-stage Run button) —
  cleanly tears down the Docker containers. The worker thread exits as soon
  as its subprocess returns; the in-process thread itself remains until
  then. Recommended approach.
* **Restart kernel** (`Kernel → Restart` in JupyterLab) — kills the
  in-process daemon threads and any *foreground* container, but detached
  containers (LS-DYNA via `start_new_session=True`) survive. Follow up with
  Stop containers, or `docker compose -p lsdyna_<run> down` in a shell.

For manual command-line aborts see Section 6 below.

In [ ]:
runners = v.make_stage_runners(rp, rc)

### 4.0 New run folder

Creates `data/runs/<timestamp>_<CABLE>/`, writes the cable preset selection to
`cable_parameters_user.json`, and selects the new folder in the picker above.

In [ ]:
runners['create_folder'].display()

### 4.1 Cable parameters + cross-section schematic

`run_cable_parameters` rewrites `cable_parameters.json` for the configured
cable + termination time. The cell below renders the *target* cross-section
from the JSON (the rectangle FreeCAD + LS-DYNA aim at).

In [ ]:
runners['cable_params'].display()
runners['metadata'].display()

In [ ]:
if rp.run_folder is None:
    print('No run folder yet.')
else:
    cable_params = v.read_cable_params(rp.run_folder)
    if cable_params:
        print({k: cable_params[k] for k in (
            'cable_name', 'cable_width', 'cable_height',
            'N_Strands', 'D_Strand_base', 'T_pitch') if k in cable_params})
        v.plot_cable_schematic(cable_params);
    else:
        print('No cable_parameters.json yet - click "1. Cable parameters" above.')

### 4.2 FreeCAD STEP geometry

Drives `tools/freecad/.../freecadcmd.exe` headless to write `<cable>.step`.
Takes ~30 s on a laptop. No live progress (FreeCAD is silent until done).

In [ ]:
runners['step'].display()

### 4.3 LS-DYNA mesh + mesh conversion

Two sub-stages:

* **3a. LS-DYNA mesh (Ansys Mechanical)** â€” pulls/uses `mechanical:25.2`
  Docker image to mesh the STEP into `LSDYNA/mesh.k`. ~5 min.
* **3b. Mesh conversion (.k)** â€” stitches mesh.k into `processed_input.k`
  via `inputfile_generator`. Seconds.

In [ ]:
runners['lsdyna_setup'].display()
runners['mesh_conv'].display()

### 4.4 LS-DYNA solve (heavy)

Spawns the `lsdyna:25.2` container via `docker compose up` and waits for
completion. The Output panel tails `<run>/LSDYNA/lsdyna_container.log` so you
see the LS-DYNA percent live. Multi-hour run for production
parameters; minutes for `termination_time = 1e-4 ms` (default).

In [ ]:
runners['lsdyna_solve'].display()

### 4.5 ParaView strand extraction + deformed-strand plot

`pvpython extract_coordinates_stack_sort.py` reads the d3plot and writes one
CSV per strand to `<run>/stack/`. The cell below overlays them on the cable
rectangle (set `STACK=<int>` for a single stack).

In [ ]:
runners['paraview'].display()

In [ ]:
STACK = None  # 1..n_stacks to isolate one
if rp.run_folder is None:
    print('No run folder.')
else:
    try:
        v.plot_deformed_strands(rp.run_folder, stack=STACK);
    except FileNotFoundError as e:
        print(e)

### 4.6 APDL conformal submodel + mesh viz

Builds the conformal mesh on deformed strands and emits the APDL `.inp`
fragments under `apdl_runfolder/`. The SVG plots under `plots/` are the
geometry the cablestack solver will use.

In [ ]:
runners['apdl_submodel'].display()

In [ ]:
if rp.run_folder is None:
    print('No run folder.')
else:
    plots = v.find_cablestack_plots(rp.run_folder)
    v.show_svgs(plots['conformal_mesh'][:3], header='Conformal mesh (top 3)')

### 4.8 Cablestack (heavy)

Copies + patches the APDL templates and launches the four-stage solve. On
local Docker, parallel up to `cablestack.max_parallel_stages` MAPDL
containers. With the HPC toggle on, uploads the runfolder, submits a SLURM
job, and waits.

Loading schedule appears below once `loading_cycle.json` is written.

In [ ]:
runners['cablestack'].display()

In [ ]:
if rp.run_folder is None:
    print('No run folder.')
else:
    lc = v.read_loading_cycle(rp.run_folder)
    if not lc:
        print('No loading_cycle.json yet - run the cablestack copy/patch step.')
    else:
        print(f"cable: {lc.get('cable')}  n_strands={lc.get('n_strands')} "
              f"n_stacks={lc.get('n_stacks')}")
        print(f"peak_force_N={lc.get('peak_force_N')}  min_force_N={lc.get('min_force_N')}")
        print(f"ramp_pressures_MPa={lc.get('ramp_pressures_MPa')}")
        print(f"\n{'step':>5} {'time':>6} {'P [MPa]':>10}  kind     label")
        for s in lc.get('steps', []):
            print(f"{s['index']:>5} {s['time']:>6.2f} {s['pressure_MPa']:>10.4f}  "
                  f"{s['kind']:<8} {s.get('label','')}")

In [ ]:
if rp.run_folder is None:
    print('No run folder.')
else:
    plots = v.find_cablestack_plots(rp.run_folder)
    v.show_svgs(plots['postprocess'], header='Cablestack postprocess')
    try:
        v.plot_stress_strain(rp.run_folder);
    except FileNotFoundError as e:
        print(e)

### 4.9 Compression box (opt-in)

Heavy 3D parent + 2D submodel solve. Only enable when
`compression_box.enabled = true` in `cable_parameters_user.json` or you tick
the `--compbox` toggle in the run-config panel.

In [ ]:
runners['compbox'].display()

## 5. Re-run cablestack postprocess (idempotent)

Re-runs `analyse_pressure` on the selected run folder without touching the
MAPDL solve. Useful after editing the analysis scripts.

In [ ]:
def rerun_postprocess(stage_name=None):
    rf = rp.run_folder
    if rf is None:
        print('Pick a run folder first.')
        return
    ok = v.run_postprocess(rf, stage_name=stage_name)
    print('Postprocess result:', ok)

# rerun_postprocess()                                # all four stages
# rerun_postprocess('pressure_radial')                # one stage

## 6. Stop / abort a running or failing container

Heavy stages (LS-DYNA, cablestack, compbox) launch Docker containers that may
detach from the kernel (LS-DYNA does this deliberately so the orchestrator can
continue while the solve runs). If a stage hangs, gets stuck on a license
error, or you just want to abort, use the helpers below.

**From the notebook (selected run folder):**

```python
# List every container project tied to the current run folder
v.list_run_containers(rp.run_folder)

# Tear them all down (docker compose -p <project> down --remove-orphans)
v.stop_run_containers(rp.run_folder)

# Or print the equivalent shell commands for manual use
print(v.container_abort_commands(rp.run_folder))
```

**From a shell — manual commands.** Project names follow strict conventions
(see CLAUDE.md "Docker project names"); lowercase, derived from the run
folder name:

| Container | Project name |
|---|---|
| LS-DYNA solve | `lsdyna_<run_folder>` |
| Ansys Mechanical mesher | `ansys_mechanical_<run_folder>` |
| Cablestack MAPDL stage | `mapdl_<run_folder>_<stage_name>` |
| Compression-box MAPDL | `mapdl_<run_folder>_compbox_<parent_mag\|submodel>` |

```bash
# Stop one
docker compose -p lsdyna_<run_folder> down

# Find every running project for this run folder
docker ps --filter label=com.docker.compose.project \
  --format '{{.Names}}  {{.Labels}}' | grep '<run_folder>'

# Nuclear option: kill every MAPDL container on this host
docker ps -q --filter "name=mapdl_" | xargs -r docker stop
```

**What about the in-process thread?** The Stop button kills the *container*
but not the daemon thread that's blocked on its subprocess. The thread exits
on its own once the subprocess returns (which it does the moment `docker
compose down` removes the container). If you need to free the thread
immediately, also restart the kernel (`Kernel → Restart`).

**HPC jobs.** If `--hpc` / Use HPC was on, the solve is a SLURM job on the
remote cluster, not a local Docker container. Cancel with `ssh <host> scancel
<JOBID>` (the JOBID is in the cablestack stage's Output panel).

In [ ]:
import ipywidgets as W
from IPython.display import display

_abort_btn = W.Button(description='Stop ALL containers for current run',
                      button_style='danger',
                      layout=W.Layout(width='320px'))
_abort_status = W.HTML(value='<i>idle</i>')
_abort_out = W.Output(layout=W.Layout(border='1px solid #ccc',
                                      max_height='240px', overflow='auto',
                                      padding='4px'))

def _do_abort(_b):
    _abort_out.clear_output()
    with _abort_out:
        rf = rp.run_folder
        if rf is None:
            print('No run folder selected.')
            return
        print(f'docker compose down for run folder: {rf.name}')
        results = v.stop_run_containers(rf)
        print(f'results: {results}')
    _abort_status.value = '<span style="color:#2e7d32;font-family:monospace">done</span>'

_abort_btn.on_click(_do_abort)
display(W.VBox([W.HBox([_abort_btn, _abort_status]), _abort_out]))

## Compatibility notes

* Every stage button calls a `WorkflowRunner` method directly. New flags or
  changed signatures need updating in `RunConfig.kwargs_for_run_workflow` and
  the closures inside `make_stage_runners` (both in `workflow_viz.py`).
* The stage tracker reads `metadata.json`'s `workflow_steps`; new keys show
  up once added to `STAGE_LABELS` in `workflow_viz.py`.
* HPC backend uses cluster-neutral `HPC_{USER,HOST,REMOTE_BASE}` env vars
  and works with any SSH-reachable SLURM cluster (CERN, PSI, your own).
  Default values target ETH Euler.
* Backgrounded stages run in daemon threads inside the kernel. The Docker
  containers they launch outlive the kernel (LS-DYNA detaches). To stop them
  cleanly, use the **Stop containers for this run** button below, or
  `docker compose -p <project> down` from a shell.